# Combined Dataset
Performing analysis on the combined dataset consists of 98 repos from 5 benchmark datasets

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Configuration
# Path the consolidated CSV files. This file will be read and then overwritten
DATA_CSV_PATH = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/obj1_rd3.csv"
tercile_values = {}

In [4]:
def calculate_project_size(df):
    '''
    Calculates project_size ('small', 'medium', 'high') based on LoC terciles, grouped by langauge.
    '''
    print("Calculating 'project_size'...")

    def categorize(group):
        # Calculate the tercile boundaries for the 'LoC' column of the group
        tercile_1 = group['LoC'].quantile(1/3)
        tercile_2 = group['LoC'].quantile(2/3)

        language_name = group.name
        tercile_values[language_name] = {
            'small_medium_boundary': tercile_1,
            'medium_high_boundary': tercile_2
        }

        def assign_category(loc):
            if loc <= tercile_1:
                return 'small'
            elif loc <= tercile_2:
                return 'medium'
            else:
                return 'high'
        
        group['project_size'] = group['LoC'].apply(assign_category)
        return group
    
    # Apply the categorization function to each language group
    df = df.groupby('language', group_keys=False).apply(categorize)
    return df

In [5]:
def calculate_project_age(df, cutoff_year=2019):
    '''
    Calculates project_age ('new', 'mid-era') based on the median_bug_year and a specified cutoff year.
    '''

    def assign_age(year):
        if pd.isna(year):
            return 'unknown'
        return 'new' if year >= cutoff_year else 'mid-era'
    
    df['project_age'] = df['median_bug_year'].apply(assign_age)
    return df

In [6]:
def clean_dependencies(df):
    '''
    Cleans the num_dependencies column by replacing 0 values with the median of that repository's langauge group.
    '''
    print("Cleaning 'num_dependencies' column...")

    # Use transform to get the median for each language and align it with the original index
    lang_median = df.groupby('language')['num_dependencies'].transform('median')

    # Replace 0s with the calculated language-specific median
    df['num_dependencies'] = df['num_dependencies'].replace(0, np.nan).fillna(lang_median)

    return df


In [7]:
# Data prcoessing

try:
    # Load the dataset
    df = pd.read_csv(DATA_CSV_PATH)
    print(f"Successfully loaded {len(df)} records from '{DATA_CSV_PATH}'.")
except FileNotFoundError:
    print(f"Error: input file not found at '{DATA_CSV_PATH}'")
    

# Apply all three processing steps
df = calculate_project_size(df)
df = calculate_project_age(df)
df = clean_dependencies(df)

# Save the update DataFrame back to the same file
df.to_csv(DATA_CSV_PATH, index=False)

# Save the update DataFrame back to the same file
df.to_csv(DATA_CSV_PATH, index=False)

print("...Process COmplete....")
print(f"Updated Data Saved back to '{DATA_CSV_PATH}'")

# display a sample of the updated data
print("Sample of the final output:")
print(df[['repo_name', 'language', 'LoC', 'project_size', 'median_bug_year', 'project_age', 'num_dependencies']].head().to_string())

print("\n\n--- Calculated Tercile Boundaries (LoC) per Language ---")
for language, values in tercile_values.items():
    print(f"\nLanguage: {language}")
    print(f"  - 'small' <= {values['small_medium_boundary']:,.0f}")
    print(f"  - 'medium' <= {values['medium_high_boundary']:,.0f}")
    print(f"  - 'high' > {values['medium_high_boundary']:,.0f}")

Successfully loaded 98 records from '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/obj1_rd3.csv'.
Calculating 'project_size'...
Cleaning 'num_dependencies' column...
...Process COmplete....
Updated Data Saved back to '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/obj1_rd3.csv'
Sample of the final output:
                                      repo_name language      LoC project_size  median_bug_year project_age  num_dependencies
0                             quarkusio/quarkus     Java  1395911         high           2021.0         new          138257.0
1                       hannah-sten/texify-idea   Kotlin   130304         high           2021.0         new             328.0
2  horizontalsystems/unstoppable-wallet-android   Kotlin    39442        small           2019.0         new             191.0
3                   intellij-rust/intellij-rust   Kotlin   334728         high           2020.0         new     

/tmp/ipykernel_221399/725833137.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('language', group_keys=False).apply(categorize)
